In [7]:
import pandas as pd
import numpy as np

In [8]:
PATH_TO_PHQ8 = "../data_cleaned/phq8_data.csv"
PATH_TO_SLEEP = "../data_cleaned/sleep_data.csv"
PATH_TO_DEMOGRAPHICS = "../data_cleaned/totaldataset.parquet"
PATH_TO_FINAL_DATASET = "../data_cleaned/final_dataset.parquet"

In [34]:

def create_window(phq8, window_size):
    window = phq8[['participant_name', 'timeInterval_ID']].copy()

    window = window.loc[window.index.repeat(window_size)].copy()
    window['lag'] = list(range(1, window_size+1)) * len(phq8)

    # Compute the prior day IDs
    window['timeInterval_ID_window'] = window['timeInterval_ID'] - window['lag']
    return window

In [35]:
def merge_with_sleep(phq8, sleep, window_size, missing_threshold):
    window = create_window(phq8, window_size)

    sleep = sleep.rename(columns = {'timeInterval_ID': 'timeInterval_ID_sleep'})

    sleep['observed'] = 1

    window = window.merge(
        sleep,
        left_on=['participant_name', 'timeInterval_ID_window'],
        right_on=['participant_name', 'timeInterval_ID_sleep'],
        how='left'
    )

    window['missing'] = window['observed'].isna().astype(int)

    missing_counts = (
        window.groupby(['participant_name', 'timeInterval_ID'])
        ['missing']
        .sum()
        .reset_index(name=f'sleep_missing_last{window_size}')
    )

    # Define the grouping columns and identify the target columns
    group_cols = ['participant_name', 'timeInterval_ID']
    missingness_cols = ['missing', 'observed']
    window_cols = ['timeInterval_ID_window', 'lag', 'timeInterval_ID_sleep']
    irrelevant_cols = group_cols + missingness_cols + window_cols
    target_cols = [col for col in window.columns if col not in irrelevant_cols]

    # Define the base aggregation functions
    # (Using named functions for quantiles so the column names look nice later)
    funcs = [
        'min', 'max', 'mean', 'median', 'std',
        ('q25', lambda x: x.quantile(0.25)),
        ('q75', lambda x: x.quantile(0.75))
    ]

    # Perform the groupby and aggregation
    agg = window.groupby(group_cols)[target_cols].agg(funcs)

    # Flatten the MultiIndex columns (e.g., ('total_sleep_time', 'mean') -> 'total_sleep_time_mean')
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
    agg = agg.reset_index()

    #Loop through target columns to compute IQR and drop intermediate quartiles
    for col in target_cols:
        q75_col = f"{col}_q75"
        q25_col = f"{col}_q25"
        
        # Compute IQR
        agg[f"{col}_iqr"] = agg[q75_col] - agg[q25_col]
        
        # Drop the 25th and 75th percentile columns
        agg = agg.drop(columns=[q25_col, q75_col])


    phq8_with_sleep_stats = phq8.merge(
        agg,
        on=['participant_name', 'timeInterval_ID'],
        how='left'
    ).merge(
        missing_counts,
        on=['participant_name', 'timeInterval_ID'],
        how='left'
    )

    invalid_mask = phq8_with_sleep_stats[f'sleep_missing_last{window_size}'] > missing_threshold

    stat_cols = [col for col in agg.columns if col not in ['participant_name', 'timeInterval_ID']]

    phq8_with_sleep_stats.loc[invalid_mask, stat_cols] = np.nan

    return phq8_with_sleep_stats

In [36]:
phq8 = pd.read_csv(PATH_TO_PHQ8)
sleep = pd.read_csv(PATH_TO_SLEEP)
demographics = pd.read_parquet(PATH_TO_DEMOGRAPHICS)


In [37]:
phq8.head()

,participant_name,timeInterval_ID,phq8_score
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11


In [38]:
sleep.head()

,participant_name,timeInterval_ID,total_sleep_time,time_in_bed,light_pct,deep_pct,REM_pct,awake_pct,NREM_pct,sleep_onset,sleep_offset,rem_latency,sleep_efficiency,awakenings,insomnia,hypersomnia
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1208,19410,21030,60.199715,17.974322,14.122682,7.703281,78.174037,3.583333,9.350000,3.025000,92.296719,1,0,0
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1209,16350,18300,45.737705,26.393443,17.213115,10.655738,72.131148,3.566667,8.633333,4.175000,89.344262,3,0,0
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1210,23490,25110,42.771804,24.731183,26.045400,6.451613,67.502987,2.800000,9.616667,2.191667,93.548387,1,0,0
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1211,21030,23040,54.296875,16.015625,20.963542,8.723958,70.312500,2.783333,9.033333,2.466667,91.276042,2,0,0
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1214,24180,26790,55.767077,21.612542,12.877940,9.742441,77.379619,2.350000,9.783333,3.208333,90.257559,2,0,0


In [39]:
window_size = 14
window = create_window(phq8, window_size)


In [40]:
window[window["timeInterval_ID"] == 1113]

,participant_name,timeInterval_ID,lag,timeInterval_ID_window
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,1,1112
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,2,1111
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,3,1110
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,4,1109
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,5,1108
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,6,1107
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,7,1106
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,8,1105
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9,1104
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,10,1103


In [41]:
sleep = sleep.rename(columns = {'timeInterval_ID': 'timeInterval_ID_sleep'})

sleep['observed'] = 1

window = window.merge(
    sleep,
    left_on=['participant_name', 'timeInterval_ID_window'],
    right_on=['participant_name', 'timeInterval_ID_sleep'],
    how='left'
)

In [42]:
window

,participant_name,timeInterval_ID,lag,timeInterval_ID_window,timeInterval_ID_sleep,total_sleep_time,time_in_bed,light_pct,deep_pct,REM_pct,awake_pct,NREM_pct,sleep_onset,sleep_offset,rem_latency,sleep_efficiency,awakenings,insomnia,hypersomnia,observed
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,1,1112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,2,1111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,3,1110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,4,1109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,5,1108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183703,5e836448-4f14-468d-9ab8-18189c169fca,1496,10,1486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
183704,5e836448-4f14-468d-9ab8-18189c169fca,1496,11,1485,1485.0,13620.0,14220.0,86.075949,6.962025,2.742616,4.219409,93.037975,23.116667,3.050000,1.708333,95.780591,1.0,0.0,0.0,1.0
183705,5e836448-4f14-468d-9ab8-18189c169fca,1496,12,1484,1484.0,10290.0,10920.0,79.945055,3.021978,11.263736,5.769231,82.967033,21.350000,0.366667,1.625000,94.230769,1.0,0.0,0.0,1.0
183706,5e836448-4f14-468d-9ab8-18189c169fca,1496,13,1483,1483.0,17040.0,19350.0,72.713178,1.550388,13.798450,11.937984,74.263566,1.733333,7.100000,3.816667,88.062016,2.0,0.0,0.0,1.0


In [43]:
window['missing'] = window['observed'].isna().astype(int)


In [44]:
missing_counts = (
    window.groupby(['participant_name', 'timeInterval_ID'])
    ['missing']
    .sum()
    .reset_index(name=f'sleep_missing_last{window_size}')
)

In [45]:
missing_counts

,participant_name,timeInterval_ID,sleep_missing_last14
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,14
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1127,14
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1141,14
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1169,14
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1197,14
...,...,...,...
13117,ffd0fbde-c320-42d6-80a1-a7656da661dd,1191,1
13118,ffd0fbde-c320-42d6-80a1-a7656da661dd,1275,1
13119,ffd0fbde-c320-42d6-80a1-a7656da661dd,1289,5
13120,ffd0fbde-c320-42d6-80a1-a7656da661dd,1445,2


In [46]:
group_cols = ['participant_name', 'timeInterval_ID']
missingness_cols = ['missing', 'observed']
window_cols = ['timeInterval_ID_window', 'lag', 'timeInterval_ID_sleep']
irrelevant_cols = group_cols + missingness_cols + window_cols
target_cols = [col for col in window.columns if col not in irrelevant_cols]

# Define the base aggregation functions
# (Using named functions for quantiles so the column names look nice later)
funcs = [
    'min', 'max', 'mean', 'median', 'std',
    ('q25', lambda x: x.quantile(0.25)),
    ('q75', lambda x: x.quantile(0.75))
]

# Perform the groupby and aggregation
agg = window.groupby(group_cols)[target_cols].agg(funcs)

In [47]:
agg

total_sleep_time  \
                                                                  min   
participant_name                     timeInterval_ID                    
007751c5-d7ad-4bec-a58f-abf32500e2ae 1113                         NaN   
                                     1127                         NaN   
                                     1141                         NaN   
                                     1169                         NaN   
                                     1197                         NaN   
...                                                               ...   
ffd0fbde-c320-42d6-80a1-a7656da661dd 1191                     26220.0   
                                     1275                     26550.0   
                                     1289                     24780.0   
                                     1445                     27330.0   
                                     1485                     33510.0   

                                                                             \
                                                          max          mean   
participant_name                     timeInterval_ID                          
007751c5-d7ad-4bec-a58f-abf32500e2ae 1113                 NaN           NaN   
                                     1127                 NaN           NaN   
                                     1141                 NaN           NaN   
                                     1169                 NaN           NaN   
                                     1197                 NaN           NaN   
...                                                       ...           ...   
ffd0fbde-c320-42d6-80a1-a7656da661dd 1191             44730.0  34234.615385   
                                     1275             35940.0  30286.153846   
                                     1289             41160.0  29866.666667   
                                     1445             39870.0  33405.000000   
                                     1485             35430.0  34370.000000   

                                                                            \
                                                       median          std   
participant_name                     timeInterval_ID                         
007751c5-d7ad-4bec-a58f-abf32500e2ae 1113                 NaN          NaN   
                                     1127                 NaN          NaN   
                                     1141                 NaN          NaN   
                                     1169                 NaN          NaN   
                                     1197                 NaN          NaN   
...                                                       ...          ...   
ffd0fbde-c320-42d6-80a1-a7656da661dd 1191             33720.0  5267.611121   
                                     1275             30030.0  2869.136161   
                                     1289             28920.0  4512.709829   
                                     1445             32880.0  3711.709971   
                                     1485             34170.0   975.499872   

                                                                        \
                                                          q25      q75   
participant_name                     timeInterval_ID                     
007751c5-d7ad-4bec-a58f-abf32500e2ae 1113                 NaN      NaN   
                                     1127                 NaN      NaN   
                                     1141                 NaN      NaN   
                                     1169                 NaN      NaN   
                                     1197                 NaN      NaN   
...                                                       ...      ...   
ffd0fbde-c320-42d6-80a1-a7656da661dd 1191             30210.0  35640.0   
                                     1275             28260.0  31710.0   
                

In [48]:
agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
agg = agg.reset_index()

In [49]:
agg

,participant_name,timeInterval_ID,total_sleep_time_min,total_sleep_time_max,total_sleep_time_mean,total_sleep_time_median,total_sleep_time_std,total_sleep_time_q25,total_sleep_time_q75,time_in_bed_min,...,insomnia_std,insomnia_q25,insomnia_q75,hypersomnia_min,hypersomnia_max,hypersomnia_mean,hypersomnia_median,hypersomnia_std,hypersomnia_q25,hypersomnia_q75
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13117,ffd0fbde-c320-42d6-80a1-a7656da661dd,1191,26220.0,44730.0,34234.615385,33720.0,5267.611121,30210.0,35640.0,31560.0,...,0.0,0.0,0.0,0.0,1.0,0.230769,0.0,0.438529,0.0,0.0
13118,ffd0fbde-c320-42d6-80a1-a7656da661dd,1275,26550.0,35940.0,30286.153846,30030.0,2869.136161,28260.0,31710.0,30270.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
13119,ffd0fbde-c320-42d6-80a1-a7656da661dd,1289,24780.0,41160.0,29866.666667,28920.0,4512.709829,28650.0,29520.0,28890.0,...,0.0,0.0,0.0,0.0,1.0,0.111111,0.0,0.333333,0.0,0.0
13120,ffd0fbde-c320-42d6-80a1-a7656da661dd,1445,27330.0,39870.0,33405.000000,32880.0,3711.709971,30855.0,36645.0,32340.0,...,0.0,0.0,0.0,0.0,1.0,0.333333,0.0,0.492366,0.0,1.0


In [50]:
for col in target_cols:
    q75_col = f"{col}_q75"
    q25_col = f"{col}_q25"
    
    # Compute IQR
    agg[f"{col}_iqr"] = agg[q75_col] - agg[q25_col]
    
    # Drop the 25th and 75th percentile columns
    agg = agg.drop(columns=[q25_col, q75_col])

In [53]:
hypersomnia_insomnia_sum = window.groupby(group_cols)[["hypersomnia", "insomnia"]].agg('sum')


In [54]:
hypersomnia_insomnia_sum

hypersomnia  insomnia
participant_name                     timeInterval_ID                       
007751c5-d7ad-4bec-a58f-abf32500e2ae 1113                     0.0       0.0
                                     1127                     0.0       0.0
                                     1141                     0.0       0.0
                                     1169                     0.0       0.0
                                     1197                     0.0       0.0
...                                                           ...       ...
ffd0fbde-c320-42d6-80a1-a7656da661dd 1191                     3.0       0.0
                                     1275                     0.0       0.0
                                     1289                     1.0       0.0
                                     1445                     4.0       0.0
                                     1485                     0.0       0.0

[13122 rows x 2 columns]

In [56]:
hypersomnia_insomnia_sum = hypersomnia_insomnia_sum.reset_index()


In [57]:
hypersomnia_insomnia_sum

,participant_name,timeInterval_ID,hypersomnia,insomnia
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,0.0,0.0
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1127,0.0,0.0
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1141,0.0,0.0
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1169,0.0,0.0
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1197,0.0,0.0
...,...,...,...,...
13117,ffd0fbde-c320-42d6-80a1-a7656da661dd,1191,3.0,0.0
13118,ffd0fbde-c320-42d6-80a1-a7656da661dd,1275,0.0,0.0
13119,ffd0fbde-c320-42d6-80a1-a7656da661dd,1289,1.0,0.0
13120,ffd0fbde-c320-42d6-80a1-a7656da661dd,1445,4.0,0.0


In [21]:
agg

,participant_name,timeInterval_ID,total_sleep_time_min,total_sleep_time_max,total_sleep_time_mean,total_sleep_time_median,total_sleep_time_std,time_in_bed_min,time_in_bed_max,time_in_bed_mean,...,REM_pct_iqr,awake_pct_iqr,NREM_pct_iqr,sleep_onset_iqr,sleep_offset_iqr,rem_latency_iqr,sleep_efficiency_iqr,awakenings_iqr,insomnia_iqr,hypersomnia_iqr
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,007751c5-d7ad-4bec-a58f-abf32500e2ae,1127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,007751c5-d7ad-4bec-a58f-abf32500e2ae,1141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,007751c5-d7ad-4bec-a58f-abf32500e2ae,1169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,007751c5-d7ad-4bec-a58f-abf32500e2ae,1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13117,ffd0fbde-c320-42d6-80a1-a7656da661dd,1191,26220.0,44730.0,34234.615385,33720.0,5267.611121,31560.0,52710.0,40001.538462,...,6.053895,4.075074,11.024294,1.600000,0.733333,1.725000,4.075074,2.00,0.0,0.0
13118,ffd0fbde-c320-42d6-80a1-a7656da661dd,1275,26550.0,35940.0,30286.153846,30030.0,2869.136161,30270.0,42810.0,34626.923077,...,6.524063,2.401175,5.578053,0.483333,0.516667,1.825000,2.401175,1.00,0.0,0.0
13119,ffd0fbde-c320-42d6-80a1-a7656da661dd,1289,24780.0,41160.0,29866.666667,28920.0,4512.709829,28890.0,44400.0,33503.333333,...,6.978711,4.401459,3.462776,0.300000,0.133333,1.508333,4.401459,2.00,0.0,0.0
13120,ffd0fbde-c320-42d6-80a1-a7656da661dd,1445,27330.0,39870.0,33405.000000,32880.0,3711.709971,32340.0,44940.0,38272.500000,...,10.010924,5.270960,7.204968,1.108333,1.420833,0.379167,5.270960,3.25,0.0,1.0


In [22]:
phq8_with_sleep_stats = phq8.merge(
    agg,
    on=['participant_name', 'timeInterval_ID'],
    how='left'
).merge(
    missing_counts,
    on=['participant_name', 'timeInterval_ID'],
    how='left'
)

In [23]:
phq8_with_sleep_stats

,participant_name,timeInterval_ID,phq8_score,total_sleep_time_min,total_sleep_time_max,total_sleep_time_mean,total_sleep_time_median,total_sleep_time_std,time_in_bed_min,time_in_bed_max,...,awake_pct_iqr,NREM_pct_iqr,sleep_onset_iqr,sleep_offset_iqr,rem_latency_iqr,sleep_efficiency_iqr,awakenings_iqr,insomnia_iqr,hypersomnia_iqr,sleep_missing_last14
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1,10560.0,30390.0,24657.000000,26070.0,6000.957424,11100.0,33150.0,...,3.653831,11.344583,0.900000,1.175000,0.500000,3.653831,0.00,0.0,0.0,4
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1,18240.0,26880.0,23061.818182,23490.0,2342.156349,20430.0,28140.0,...,3.724093,7.770862,0.666667,0.508333,1.208333,3.724093,0.50,0.0,0.0,3
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3,12930.0,29220.0,21750.000000,21210.0,5975.707071,13410.0,31770.0,...,4.334132,2.954602,20.616667,1.200000,1.458333,4.334132,2.00,0.0,0.0,5
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11,22410.0,30840.0,24747.000000,23535.0,2893.732883,24390.0,33240.0,...,3.484294,8.297179,1.037500,0.862500,1.025000,3.484294,2.75,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13117,5f3217f1-c751-4fee-a8c4-21acc5153d58,1831,20,27090.0,32400.0,29740.714286,29460.0,1699.201008,28530.0,34200.0,...,3.659095,10.473600,0.250000,0.737500,2.325000,3.659095,1.00,0.0,0.0,0
13118,5f3217f1-c751-4fee-a8c4-21acc5153d58,1845,20,27990.0,36420.0,32860.714286,32925.0,2713.086418,30210.0,38790.0,...,2.485481,5.328105,0.416667,0.945833,1.018750,2.485481,1.00,0.0,0.0,0
13119,5f3217f1-c751-4fee-a8c4-21acc5153d58,1859,17,28470.0,36630.0,32252.307692,32070.0,1996.351480,30780.0,39450.0,...,3.114725,5.159355,0.350000,0.583333,0.983333,3.114725,1.00,0.0,0.0,1
13120,6b52c798-bf6e-4512-af0d-98e39939fd37,1642,8,12660.0,33600.0,20850.000000,19770.0,6953.363215,13080.0,36090.0,...,5.606985,3.807482,22.000000,2.041667,0.687500,5.606985,2.50,0.0,0.0,3


In [24]:
phq8_with_sleep_stats.columns

Index(['participant_name', 'timeInterval_ID', 'phq8_score',
       'total_sleep_time_min', 'total_sleep_time_max', 'total_sleep_time_mean',
       'total_sleep_time_median', 'total_sleep_time_std', 'time_in_bed_min',
       'time_in_bed_max', 'time_in_bed_mean', 'time_in_bed_median',
       'time_in_bed_std', 'light_pct_min', 'light_pct_max', 'light_pct_mean',
       'light_pct_median', 'light_pct_std', 'deep_pct_min', 'deep_pct_max',
       'deep_pct_mean', 'deep_pct_median', 'deep_pct_std', 'REM_pct_min',
       'REM_pct_max', 'REM_pct_mean', 'REM_pct_median', 'REM_pct_std',
       'awake_pct_min', 'awake_pct_max', 'awake_pct_mean', 'awake_pct_median',
       'awake_pct_std', 'NREM_pct_min', 'NREM_pct_max', 'NREM_pct_mean',
       'NREM_pct_median', 'NREM_pct_std', 'sleep_onset_min', 'sleep_onset_max',
       'sleep_onset_mean', 'sleep_onset_median', 'sleep_onset_std',
       'sleep_offset_min', 'sleep_offset_max', 'sleep_offset_mean',
       'sleep_offset_median', 'sleep_offset_std'

In [25]:
agg.columns

Index(['participant_name', 'timeInterval_ID', 'total_sleep_time_min',
       'total_sleep_time_max', 'total_sleep_time_mean',
       'total_sleep_time_median', 'total_sleep_time_std', 'time_in_bed_min',
       'time_in_bed_max', 'time_in_bed_mean', 'time_in_bed_median',
       'time_in_bed_std', 'light_pct_min', 'light_pct_max', 'light_pct_mean',
       'light_pct_median', 'light_pct_std', 'deep_pct_min', 'deep_pct_max',
       'deep_pct_mean', 'deep_pct_median', 'deep_pct_std', 'REM_pct_min',
       'REM_pct_max', 'REM_pct_mean', 'REM_pct_median', 'REM_pct_std',
       'awake_pct_min', 'awake_pct_max', 'awake_pct_mean', 'awake_pct_median',
       'awake_pct_std', 'NREM_pct_min', 'NREM_pct_max', 'NREM_pct_mean',
       'NREM_pct_median', 'NREM_pct_std', 'sleep_onset_min', 'sleep_onset_max',
       'sleep_onset_mean', 'sleep_onset_median', 'sleep_onset_std',
       'sleep_offset_min', 'sleep_offset_max', 'sleep_offset_mean',
       'sleep_offset_median', 'sleep_offset_std', 'rem_latency

In [26]:
missing_threshold = 7

In [27]:
invalid_mask = phq8_with_sleep_stats[f'sleep_missing_last{window_size}'] > missing_threshold

stat_cols = [col for col in agg.columns if col not in ['participant_name', 'timeInterval_ID']]

phq8_with_sleep_stats.loc[invalid_mask, stat_cols] = np.nan

In [28]:
phq8_with_sleep_stats

,participant_name,timeInterval_ID,phq8_score,total_sleep_time_min,total_sleep_time_max,total_sleep_time_mean,total_sleep_time_median,total_sleep_time_std,time_in_bed_min,time_in_bed_max,...,awake_pct_iqr,NREM_pct_iqr,sleep_onset_iqr,sleep_offset_iqr,rem_latency_iqr,sleep_efficiency_iqr,awakenings_iqr,insomnia_iqr,hypersomnia_iqr,sleep_missing_last14
0,007751c5-d7ad-4bec-a58f-abf32500e2ae,1113,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1458,1,10560.0,30390.0,24657.000000,26070.0,6000.957424,11100.0,33150.0,...,3.653831,11.344583,0.900000,1.175000,0.500000,3.653831,0.00,0.0,0.0,4
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1472,1,18240.0,26880.0,23061.818182,23490.0,2342.156349,20430.0,28140.0,...,3.724093,7.770862,0.666667,0.508333,1.208333,3.724093,0.50,0.0,0.0,3
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1486,3,12930.0,29220.0,21750.000000,21210.0,5975.707071,13410.0,31770.0,...,4.334132,2.954602,20.616667,1.200000,1.458333,4.334132,2.00,0.0,0.0,5
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1514,11,22410.0,30840.0,24747.000000,23535.0,2893.732883,24390.0,33240.0,...,3.484294,8.297179,1.037500,0.862500,1.025000,3.484294,2.75,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13117,5f3217f1-c751-4fee-a8c4-21acc5153d58,1831,20,27090.0,32400.0,29740.714286,29460.0,1699.201008,28530.0,34200.0,...,3.659095,10.473600,0.250000,0.737500,2.325000,3.659095,1.00,0.0,0.0,0
13118,5f3217f1-c751-4fee-a8c4-21acc5153d58,1845,20,27990.0,36420.0,32860.714286,32925.0,2713.086418,30210.0,38790.0,...,2.485481,5.328105,0.416667,0.945833,1.018750,2.485481,1.00,0.0,0.0,0
13119,5f3217f1-c751-4fee-a8c4-21acc5153d58,1859,17,28470.0,36630.0,32252.307692,32070.0,1996.351480,30780.0,39450.0,...,3.114725,5.159355,0.350000,0.583333,0.983333,3.114725,1.00,0.0,0.0,1
13120,6b52c798-bf6e-4512-af0d-98e39939fd37,1642,8,12660.0,33600.0,20850.000000,19770.0,6953.363215,13080.0,36090.0,...,5.606985,3.807482,22.000000,2.041667,0.687500,5.606985,2.50,0.0,0.0,3


In [30]:
missingness = phq8_with_sleep_stats.isnull().mean().sort_values(ascending=False)

In [31]:
for col, missing in missingness.items():
    print(f"{col}: {missing}" )

sleep_offset_max: 0.36297820454199053
sleep_efficiency_min: 0.36297820454199053
awakenings_median: 0.36297820454199053
awakenings_mean: 0.36297820454199053
awakenings_max: 0.36297820454199053
awakenings_min: 0.36297820454199053
sleep_efficiency_std: 0.36297820454199053
sleep_efficiency_median: 0.36297820454199053
sleep_efficiency_mean: 0.36297820454199053
sleep_efficiency_max: 0.36297820454199053
rem_latency_std: 0.36297820454199053
awakenings_iqr: 0.36297820454199053
rem_latency_median: 0.36297820454199053
rem_latency_mean: 0.36297820454199053
rem_latency_max: 0.36297820454199053
rem_latency_min: 0.36297820454199053
sleep_offset_std: 0.36297820454199053
sleep_offset_median: 0.36297820454199053
sleep_offset_mean: 0.36297820454199053
insomnia_iqr: 0.36297820454199053
awakenings_std: 0.36297820454199053
insomnia_min: 0.36297820454199053
insomnia_max: 0.36297820454199053
insomnia_mean: 0.36297820454199053
rem_latency_iqr: 0.36297820454199053
sleep_offset_iqr: 0.36297820454199053
sleep_ons

In [ ]:
window_size = 14
missing_threshold = 7

#phq8_missingness_adjusted = remove_missingness(phq8, sleep, window_size, missing_threshold)

phq8_merged_with_sleep = merge_with_sleep(phq8, sleep, window_size, missing_threshold)

In [9]:
final_data = pd.read_parquet(PATH_TO_FINAL_DATASET)

In [11]:
for col in final_data.columns:
    print(col)

participant_name
timeInterval_ID
phq8_score
total_sleep_time_min
total_sleep_time_max
total_sleep_time_mean
total_sleep_time_median
total_sleep_time_std
total_sleep_time_sum
total_sleep_time_pct_short_sleep
total_sleep_time_cv
time_in_bed_min
time_in_bed_max
time_in_bed_mean
time_in_bed_median
time_in_bed_std
time_in_bed_sum
time_in_bed_cv
light_pct_min
light_pct_max
light_pct_mean
light_pct_median
light_pct_std
deep_pct_min
deep_pct_max
deep_pct_mean
deep_pct_median
deep_pct_std
REM_pct_min
REM_pct_max
REM_pct_mean
REM_pct_median
REM_pct_std
awake_pct_min
awake_pct_max
awake_pct_mean
awake_pct_median
awake_pct_std
NREM_pct_min
NREM_pct_max
NREM_pct_mean
NREM_pct_median
NREM_pct_std
rem_deep_ratio_mean
light_restorative_ratio_mean
sleep_onset_min
sleep_onset_max
sleep_offset_min
sleep_offset_max
sleep_onset_mean
sleep_offset_mean
sleep_onset_std
sleep_offset_std
avg_time_in_bed_window
sleep_midpoint_mean
sleep_midpoint_std
sleep_phase_instability
rem_latency_min
rem_latency_max
rem_lat

In [13]:
final_data.shape

(13122, 102)